# 📈 Clase 3 — Segmentación y series de tiempo
**Maestría en Fintech — Programación para el Análisis de Datos**

**Preguntas que responde:** *¿Cómo comparo segmentos? ¿Cómo veo cómo evolucionan mis datos en el tiempo?*

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('✅ Librerías cargadas')

In [ ]:
# Los datos se descargan solos desde el repo del curso.
# No hace falta subir ningún archivo a Colab.
DATOS = 'https://raw.githubusercontent.com/camilojaure/itba-pad/main/datasets/'

df = pd.read_csv(DATOS + 'transacciones.csv')
print(f'Dataset: {len(df):,} transacciones')
df.head()


## 1. GroupBy — La navaja suiza del analista

GroupBy es la operación más poderosa de Pandas. La lógica es siempre la misma:
**agrupar → calcular → mostrar**

In [ ]:
# Gasto total por categoría
df.groupby('categoria')['monto_ars'].sum().sort_values(ascending=False).apply(lambda x: f'${x:,.0f}')

In [ ]:
# Múltiples métricas de una sola vez
resumen = df.groupby('categoria').agg(
    transacciones=('transaccion_id', 'count'),
    monto_total=('monto_ars', 'sum'),
    monto_promedio=('monto_ars', 'mean'),
    monto_max=('monto_ars', 'max')
).round(0).sort_values('monto_total', ascending=False)
resumen

In [ ]:
# Visualización: gasto total por categoría
montos = df.groupby('categoria')['monto_ars'].sum().sort_values()

plt.figure(figsize=(10,6))
bars = plt.barh(montos.index, montos.values / 1e6, color='steelblue')
plt.title('Gasto total por categoría (millones ARS)', fontsize=14)
plt.xlabel('Millones ARS')
for bar, val in zip(bars, montos.values / 1e6):
    plt.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'${val:.1f}M', va='center')
plt.tight_layout()
plt.show()

## 2. Pivot Tables — Lo que hacías en Excel, ahora en Python

In [ ]:
# Transacciones por categoría y tipo (débito/crédito)
pivot = df.pivot_table(
    values='monto_ars',
    index='categoria',
    columns='tipo',
    aggfunc='sum'
).round(0)
pivot

In [ ]:
# Heatmap del pivot — visualización poderosa para comparar
plt.figure(figsize=(8,6))
sns.heatmap(pivot / 1e6, annot=True, fmt='.1f', cmap='Blues',
            cbar_kws={'label': 'Millones ARS'})
plt.title('Gasto por categoría y tipo de tarjeta (M ARS)')
plt.tight_layout()
plt.show()

## 3. Series de Tiempo — Trabajando con fechas

En finanzas, casi todo tiene una dimensión temporal. Pandas tiene herramientas específicas para esto.

In [ ]:
# Convertir la columna fecha a tipo datetime
df['fecha'] = pd.to_datetime(df['fecha'])

# Extraer componentes de la fecha
df['mes'] = df['fecha'].dt.month
df['mes_nombre'] = df['fecha'].dt.strftime('%b')
df['dia_semana'] = df['fecha'].dt.day_name()
df['trimestre'] = df['fecha'].dt.quarter

print('Rango de fechas:')
print(f'  Desde: {df["fecha"].min().strftime("%d/%m/%Y")}')
print(f'  Hasta: {df["fecha"].max().strftime("%d/%m/%Y")}')
df[['fecha','mes','mes_nombre','dia_semana','trimestre']].head(5)

In [ ]:
# Evolución mensual del volumen de transacciones
evolucion = df.set_index('fecha').resample('ME')['monto_ars'].agg(['sum','count'])
evolucion.columns = ['monto_total', 'cant_transacciones']
evolucion

In [ ]:
# Gráfico de línea temporal
fig, ax1 = plt.subplots(figsize=(12,5))

color1 = 'steelblue'
ax1.plot(evolucion.index, evolucion['monto_total']/1e6, color=color1, linewidth=2.5, marker='o')
ax1.set_ylabel('Monto total (M ARS)', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_title('Evolución mensual de transacciones — 2024', fontsize=14)

ax2 = ax1.twinx()
color2 = 'salmon'
ax2.bar(evolucion.index, evolucion['cant_transacciones'], alpha=0.3, color=color2, width=20)
ax2.set_ylabel('Cantidad de transacciones', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

plt.tight_layout()
plt.show()

In [ ]:
# Promedio móvil — suaviza la variabilidad y muestra la tendencia
evolucion['media_movil_3m'] = evolucion['monto_total'].rolling(3).mean()

plt.figure(figsize=(12,5))
plt.plot(evolucion.index, evolucion['monto_total']/1e6, label='Monto mensual', alpha=0.5, color='steelblue')
plt.plot(evolucion.index, evolucion['media_movil_3m']/1e6, label='Media móvil 3 meses', linewidth=2.5, color='darkblue')
plt.title('Volumen de transacciones con media móvil (M ARS)', fontsize=14)
plt.ylabel('Millones ARS')
plt.legend()
plt.tight_layout()
plt.show()
print('\n💡 La media móvil suaviza el ruido y muestra la tendencia real.')

In [ ]:
# Combinando: ¿cómo evolucionó cada categoría mes a mes?
evol_cat = df.pivot_table(values='monto_ars', index=pd.Grouper(key='fecha', freq='ME'),
                          columns='categoria', aggfunc='sum') / 1e6

top4 = df.groupby('categoria')['monto_ars'].sum().nlargest(4).index
evol_cat[top4].plot(figsize=(12,5), linewidth=2)
plt.title('Evolución mensual — Top 4 categorías (M ARS)', fontsize=14)
plt.ylabel('Millones ARS')
plt.legend(title='Categoría', bbox_to_anchor=(1.01,1))
plt.tight_layout()
plt.show()

---
## 🧑‍💻 Práctica — Tu turno

In [ ]:
# EJERCICIO 1
# ¿Cuál es el monto promedio de transacción por tipo (débito vs crédito)?
# Tu código acá:


In [ ]:
# EJERCICIO 2
# ¿En qué día de la semana se gasta más? Mostralo en un gráfico de barras.
# Tu código acá:


In [ ]:
# EJERCICIO 3
# Creá un pivot table que muestre el monto promedio por categoría y trimestre.
# Tu código acá:


In [ ]:
# EJERCICIO 4
# ¿Cuántas transacciones rechazadas (aprobada == 0) hay por mes?
# Graficalo como serie de tiempo.
# Tu código acá:


In [ ]:
# EJERCICIO 5
# ¿Cuál categoría tuvo el mayor crecimiento de gasto del Q1 al Q4?
# Tu código acá:


In [ ]:
# EJERCICIO 6
# Calculá el promedio móvil de 6 meses del monto total de transacciones.
# Graficalo junto con la serie original.
# Tu código acá:


In [ ]:
# EJERCICIO 7
# ¿Qué porcentaje del gasto total representa cada categoría?
# Mostralo como un gráfico de torta (plt.pie) o barras apiladas al 100%.
# Tu código acá:


In [ ]:
# EJERCICIO 8
# Filtrá solo las transacciones de 'Viajes' y analizá su evolución mensual.
# ¿Ves alguna estacionalidad? Comentá lo que observás.
# Tu código acá:

# Observación:
# 


In [ ]:
# EJERCICIO 9
# Calculá el ticket promedio por cliente por mes.
# Tip: groupby(['cliente_id', mes]) → mean() → luego mean() por mes
# Tu código acá:


In [ ]:
# EJERCICIO 10 — Desafío
# Armá un resumen trimestral que muestre para cada trimestre:
# - monto total
# - cantidad de transacciones
# - ticket promedio
# - categoría con más volumen
# Contá en 3-4 líneas qué trimestre fue el más dinámico y por qué.
# Tu código acá:

# Tu conclusión:
# 
